In [1]:
pretty_print_default()

In [2]:
def random_polynomial(n):
    R.<x> = ZZ[]
    f = R.random_element(degree=n)
    return f

In [3]:
R.<x> = ZZ[]
R.random_element(degree=3)

-4*x^3 - x^2 + x + 1

In [4]:
def good_primes(f, N):
    r"""
    Iterator that yields C with defining polynomial f over F_p for good p up to N
    """
    for p in prime_range(3, N + 1):
        K = GF(p)
        fp = f.change_ring(K)
        if fp.is_squarefree() and fp.degree() >= 3:
            yield HyperellipticCurve(fp)

In [5]:
def compute_A_tilde_F(C, lam):
    K = C.base_ring()
    p = K.characteristic()
    assert K.degree() == 1
    g = C.genus()

    f, _ = C.hyperelliptic_polynomials()
    F = f.change_ring(Zmod(p^lam))
    H = F^((p - 1) // 2)

    for l in range(lam):
        H_l = H^(2 * l + 1)
        size = (2 * l + 1)  * (g + 1) + 1
        rows = [[H_l[v * p - u] for u in range(size)] for v in range(size)]
        M = Matrix(rows)
        assert M.dimensions() == (size, size)
        yield M

In [6]:
@cached_method
def phi_lambda(lam):
    assert lam >= 1
    R.<t> = QQ[]
    phi = R(0)
    for j in range(lam):
        term = 1 / (2^(lam + j))
        term *= binomial(lam + j - 1, j)
        term *= (1 - t)^j * (1 + t)^lam - (1 + t)^j * (1 - t)^lam
        phi += term
    return phi

In [7]:
def trace_formula(C, r, lam):
    p = C.base_ring().characteristic()
    assert lam >= 1
    assert p >= 2 * lam + 1
    R = Zmod(p^lam)
    total = R(1)
    phi = phi_lambda(lam)
    A_tilde = list(compute_A_tilde_F(C, lam))
    for l in range(lam):
        phi_coeff = phi[2 * l + 1]
        phi_coeff = R(phi_coeff.numerator()) / R(phi_coeff.denominator())
        A_tilde_l = A_tilde[l]
        total += phi_coeff * (A_tilde_l^r).trace()
    return total * (p^r - 1)

In [8]:
def count_points(C, r, lam):
    count = trace_formula(C, r, lam)
    p = C.base_ring().characteristic()
    Cpr = C.change_ring(GF(p, r))
    f, _ = Cpr.hyperelliptic_polynomials()
    g = Cpr.genus()

    if f[0] == 0:
        count += 1
    elif f[0].is_square():
        count += 2
        
    if f[2 * g + 2] == 0:
        count += 1
    elif f[2 * g + 2].is_square():
        count += 2
    return count

In [9]:
for n in range(4, 7):
    f = random_polynomial(n)
    for Cp in good_primes(f, 30):
        p = Cp.base_ring().characteristic()
        for lam in range(1, 4):
            if p < 2 * lam + 1:
                break
            assert p >= 2 * lam + 1
            max_r = 5
            my_counts = [count_points(Cp, r, lam) for r in range(1, max_r + 1)]
            sage_counts = [n % p^lam for n in Cp.count_points(max_r)]
            assert my_counts == sage_counts